# 41 · Study area's boundary repair, checked against shapely

## What this checks

**Study area** repairs a published boundary as it reads it. This check covers release 0.13.0, and that repair.

It removes two kinds of fault:

- **A repeated vertex**, a corner point equal to the one before it.
- **The tip of a spike**, where the outline goes out along a line and comes straight back, enclosing nothing.

Every removal is listed, by kind and position. On Sierra Leone's 16 districts, it found 84 repeated vertices and one
spike, in Bonthe. A repair that removed the wrong point would change the shape of a district.

## How it is checked

We repeat the repair with tools that share no code with Fieldwork.

- **Repeats** are removed by **shapely**, Python's library for shapes, which runs on GEOS, a geometry engine written
  in C++. We read what it dropped by comparing its outline with the published one.
- **Spikes** have no shapely equivalent. So the rule is written out here in Python, from the order and the test the
  data declare.
- **Areas** and whether each shape is valid come from GEOS.

The data are two sets of boundaries. One is OCHA's 16 districts of Sierra Leone. The other is geoBoundaries' 92
municipalities of Rio de Janeiro state, a set that loaded before the repair existed.

## What should happen

1. **Every repeated vertex.** The points shapely drops, outline by outline, are the repeats Fieldwork removed, in
   order.
2. **Every spike tip.** The rule written out here removes the same points as Fieldwork, in order.
3. **Every vertex count**, before and after the repair, is the same.
4. **No area changes.** Each repaired unit's area is within 10⁻¹² square degrees of the published one.

We also record, without judging, two things. One is whether GEOS finds each unit valid before and after, and why.
The other is any spike that an exact test would decide differently from the stated tolerance of 10⁻¹².

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["shapely"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_41_boundary_repair import run

report = run(Path.cwd() / "fixtures" / "boundary-repair-001.json")
print("agrees:", report["agrees"])
for level in report["levels"]:
    print(f"\n{'OK ' if level['agrees'] else 'DIFF'} {level['id']}: {level['units']} units, {level['repaired']} repaired, "
          f"{level['repeated']} repeated vertices, {level['spikes']} spike tips")
    for u in level["repairedUnits"]:
        print(f"  {u['name']}: {u['shapely']['repeated']} repeated, {len(u['shapely']['spike'])} spike; "
              f"{u['vertices']['published']} -> {u['vertices']['repaired']} vertices; area change {u['areaChange']:.1e}")
    for d in level["differs"]:
        print("  DIFF", d["name"], d["checks"])

## The result on record

First recorded on 9 November 2026, against release 0.7.0. Rerun on 26 November 2026, against release 0.13.0:
**Agrees**. This check is what counts Study area as
checked in Fieldwork's record.

- **Sierra Leone.** 14 of the 16 districts were repaired. 84 repeated vertices and 1 spike were removed, the same on
  both sides.
- **Rio de Janeiro.** None of the 92 municipalities needed repair, and none was changed.

No area changed beyond the tolerance.

Running the cell above repeats the check on the same data.

## What GEOS says

GEOS treats a repeated vertex as valid, and a spike as an outline crossing itself. So before the repair, GEOS
rejects one unit, Bonthe. Fieldwork's own check rejected fourteen. This is recorded, not judged.

In [ ]:
for level in report["levels"]:
    print(level["id"], "· GEOS invalid before:", level["geosInvalidBefore"], "· after:", level["geosInvalidAfter"] or "none")
    print("  decided differently by exact arithmetic:", level["inexact"] or "nowhere")

## What this does not show

- That the published boundaries have no other fault. An outline that crosses itself is not repaired. GEOS's verdict
  on what is left is recorded for that reason.
- That a spike is a mistake in drawing. It encloses no area, which is why it can be removed. Nothing here knows why
  it was drawn.
- How often the repair is needed on geoBoundaries in general. Only one level of one country is here.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("shapely :", report["environment"]["shapely"], "· GEOS", report["environment"]["geos"])
print("runtime :", report["environment"]["runtime"])